In [1]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [2]:
# The unique identifier for Dashen Bank's app on the Google Play Store
Dashen_APP_ID = 'com.dashen.dashensuperapp'

# Step 1: Get app metadata (rating, installs, description...)
app_info = app(
    Dashen_APP_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("Dashen Bank App Info")
print("=" * 50)
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

Dashen Bank App Info
App Title   : Dashen Bank
Current Score: 4.221034
Total Ratings: 5,611
Total Reviews: 1,025
Installs     : 1,000,000+


##  Scraping Reviews

In [3]:
# Step 2: Scrape reviews
print(f"Scraping reviews for CBE Bank...")

result, continuation_token = reviews(
    Dashen_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       # Most recent first
    count=500,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for CBE Bank...
Collected 500 raw reviews


In [4]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 88c562ed-6c0c-4a22-b33a-00b7daa48b76
  userName: Habsire
  userImage: https://play-lh.googleusercontent.com/a-/ALV-UjViNMotObNfrz6YyFDbLMN-nNz4Ek7KtNwHZlEqFd1xyUB-Lzw
  content: ምንም አይሰራም 🥹 በጣም ያናድዳል አንድ ነገር ለመጠቀማ በአስራ አምስት ቀን ሙከራ ራሱ አይሰራም በጣም የሚያስጠላ ሲስተም ነው እኔ ከአስር 1 እራሱ አልሰጠውም
  score: 1
  thumbsUpCount: 0
  reviewCreatedVersion: 1.9.14
  at: 2026-05-13 21:37:46
  replyContent: None
  repliedAt: None
  appVersion: 1.9.14


In [6]:
# Step 3: Extract only the columns we need
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'Dashen Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,88c562ed-6c0c-4a22-b33a-00b7daa48b76,ምንም አይሰራም 🥹 በጣም ያናድዳል አንድ ነገር ለመጠቀማ በአስራ አምስት ...,1,2026-05-13 21:37:46,Dashen Bank,Google Play
1,dfcafbd5-be84-4c83-814f-301c05e35491,bas,5,2026-05-13 20:58:19,Dashen Bank,Google Play
2,abf6804b-429f-4383-94a1-5a5955c27569,bad mobile banking at all,1,2026-05-13 19:14:11,Dashen Bank,Google Play
3,265cfda4-296c-4676-bc76-032559a65ec2,very nice app.,5,2026-05-13 17:44:57,Dashen Bank,Google Play
4,94a4413f-40c7-4842-b31c-43f70aff0686,very difficult app,1,2026-05-13 16:26:31,Dashen Bank,Google Play


## Exploring the Raw Data

In [7]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 500

Column dtypes:
review_id            object
review               object
rating                int64
date         datetime64[ns]
bank                 object
source               object
dtype: object


In [8]:
# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

Rating distribution:
  5 stars:  320  ████████████████████████████████████████████████████████████████
  4 stars:   33  ██████
  3 stars:   27  █████
  2 stars:   19  ███
  1 stars:  101  ████████████████████


In [9]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Sample date values (raw):
0   2026-05-13 21:37:46
1   2026-05-13 20:58:19
2   2026-05-13 19:14:11
3   2026-05-13 17:44:57
4   2026-05-13 16:26:31
5   2026-05-13 11:41:27
6   2026-05-13 08:15:58
7   2026-05-12 17:33:07
8   2026-05-12 17:22:54
9   2026-05-12 16:43:12

Date dtype: datetime64[ns]


## Data Quality Audit

In [10]:
print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

# --- Problem 1: Missing Values ---
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
  review_id      : OK
  review         : OK
  rating         : OK
  date           : OK
  bank           : OK
  source         : OK


In [11]:
# --- Problem 2: Duplicate Reviews ---
print("Problem 2: Duplicates")
print("-" * 30)

# Exact duplicates on review text
exact_dupes = df_raw.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes}")

# Duplicate review IDs
id_dupes = df_raw.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")

# Empty reviews (also a form of bad data)
empty_reviews = (df_raw['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews}")

Problem 2: Duplicates
------------------------------
  Exact duplicate reviews : 75
  Duplicate review IDs    : 0
  Empty review texts      : 0


In [12]:
# --- Problem 3: Date Format ---
print("Problem 3: Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw['date'].dtype}")
print(f"  Sample values: {df_raw['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

Problem 3: Date Format
------------------------------
  Current dtype: datetime64[ns]
  Sample values: 2026-05-13 21:37:46
  Target format: YYYY-MM-DD (string or date object)


In [13]:
# Work on a copy so raw data stays untouched
df = df_raw.copy()

print(f"Starting with: {len(df)} reviews")

Starting with: 500 reviews


### Handle Missing Values

In [14]:
before = len(df)

# Drop rows missing the critical columns
critical_cols = ['review', 'rating']
df = df.dropna(subset=critical_cols)

removed = before - len(df)
print(f"Removed {removed} rows with missing critical data")
print(f"Remaining: {len(df)} reviews")

Removed 0 rows with missing critical data
Remaining: 500 reviews


###  Remove Duplicates

In [15]:
before = len(df)

df = df.drop_duplicates(subset=['review_id'], keep='first')

removed = before - len(df)
print(f"Removed {removed} duplicate reviews")
print(f"Remaining: {len(df)} reviews")

Removed 0 duplicate reviews
Remaining: 500 reviews


###  Normalize Dates

In [16]:
print("Before normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

# Convert to pandas datetime, then format as YYYY-MM-DD string
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

print("\nAfter normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

Before normalization:
0   2026-05-13 21:37:46
1   2026-05-13 20:58:19
2   2026-05-13 19:14:11
dtype: datetime64[ns]

After normalization:
0    2026-05-13
1    2026-05-13
2    2026-05-13
dtype: object

Date range: 2025-08-13 to 2026-05-13


### Clean Review Text

In [17]:
def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text

# Show before/after on a sample review
sample_raw = "  Great   app!\n\nVery useful.  "
print(f"Before: {repr(sample_raw)}")
print(f"After : {repr(clean_text(sample_raw))}")

# Apply to the full column
df['review'] = df['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")

Before: '  Great   app!\n\nVery useful.  '
After : 'Great app! Very useful.'

Removed 0 reviews that were empty after cleaning


###  Validate Ratings

In [18]:
# Check for out-of-range ratings
invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5)]
print(f"Invalid ratings (outside 1–5): {len(invalid_ratings)}")

# Remove them
df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

# Ensure rating is stored as integer
df['rating'] = df['rating'].astype(int)

print(f"Remaining: {len(df)} reviews")
print(f"Rating dtype: {df['rating'].dtype}")

Invalid ratings (outside 1–5): 0
Remaining: 500 reviews
Rating dtype: int64


##  Final Output

In [19]:
# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (500, 5)


,review,rating,date,bank,source
0,ምንም አይሰራም 🥹 በጣም ያናድዳል አንድ ነገር ለመጠቀማ በአስራ አምስት ...,1,2026-05-13,Dashen Bank,Google Play
1,very difficult app,1,2026-05-13,Dashen Bank,Google Play
2,nice,5,2026-05-13,Dashen Bank,Google Play
3,"Good app, but debit transactions not allowed W...",5,2026-05-13,Dashen Bank,Google Play
4,bas,5,2026-05-13,Dashen Bank,Google Play
5,very nice app.,5,2026-05-13,Dashen Bank,Google Play
6,bad mobile banking at all,1,2026-05-13,Dashen Bank,Google Play
7,Very bad customer service line. they won't pic...,1,2026-05-12,Dashen Bank,Google Play
8,smart app,5,2026-05-12,Dashen Bank,Google Play
9,The application booting time is so bad..,3,2026-05-12,Dashen Bank,Google Play


In [20]:
# Save to CSV
import os
os.makedirs('data/processed', exist_ok=True)

output_path = 'data/processed/Dashen_bank_reviews_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: data/processed/Dashen_bank_reviews_clean.csv


##  Preprocessing Report

In [21]:
print("=" * 55)
print("  PREPROCESSING REPORT — CBE Bank Reviews")
print("=" * 55)

original_count = len(df_raw)
final_count    = len(df_clean)
removed_total  = original_count - final_count
retention_rate = (final_count / original_count * 100)

print(f"\n  Raw reviews collected  : {original_count:>6}")
print(f"  Reviews after cleaning : {final_count:>6}")
print(f"  Reviews removed        : {removed_total:>6}")
print(f"  Data retention rate    : {retention_rate:>5.1f}%")

quality = "EXCELLENT" if retention_rate >= 95 else ("GOOD" if retention_rate >= 90 else "NEEDS ATTENTION")
print(f"  Data quality           : {quality}")

print(f"\n  Date range : {df_clean['date'].min()}  to  {df_clean['date'].max()}")

print("\n  Rating distribution:")
for rating in sorted(df_clean['rating'].unique(), reverse=True):
    count = (df_clean['rating'] == rating).sum()
    pct   = count / final_count * 100
    bar   = '█' * (count // 5)
    print(f"    {rating} stars : {count:>4} ({pct:4.1f}%)  {bar}")

print("\n  Text length stats:")
lengths = df_clean['review'].str.len()
print(f"    Min    : {lengths.min()} characters")
print(f"    Median : {lengths.median():.0f} characters")
print(f"    Max    : {lengths.max()} characters")

print("\n  Columns in final CSV:")
for col in df_clean.columns:
    print(f"    - {col}")

print("\n" + "=" * 55)

  PREPROCESSING REPORT — CBE Bank Reviews

  Raw reviews collected  :    500
  Reviews after cleaning :    500
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2025-08-13  to  2026-05-13

  Rating distribution:
    5 stars :  320 (64.0%)  ████████████████████████████████████████████████████████████████
    4 stars :   33 ( 6.6%)  ██████
    3 stars :   27 ( 5.4%)  █████
    2 stars :   19 ( 3.8%)  ███
    1 stars :  101 (20.2%)  ████████████████████

  Text length stats:
    Min    : 1 characters
    Median : 22 characters
    Max    : 499 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source

